In [72]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score,precision_score,recall_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

In [73]:
# Agar file kisi sub-folder mein hai:
file_path = "clean/appointment_cleaned.csv"   # ya "data/omedical_appointment_cleaned.csv"

df = pd.read_csv(file_path)
print(df.shape)
df.head()

(109545, 24)


,specialty,gender,no_show,disability,appointment_shift,age,under_12_years_old,over_60_years_old,patient_needs_companion,average_temp_day,...,storm_day_before,rain_intensity,heat_intensity,Hipertension,Diabetes,Alcoholism,Handcap,Scholarship,SMS_received,appointment_datetime
0,1,0,1,0,1,9,1,0,1,23.18,...,1,1,3,0,0,0,0,0,0,2020-01-01 17:00:00
1,1,1,0,0,2,11,1,0,1,14.31,...,1,1,2,0,0,0,0,0,0,2020-01-01 07:00:00
2,2,1,0,0,1,8,1,0,1,21.61,...,1,1,3,0,0,0,0,0,0,2020-01-01 16:00:00
3,2,1,1,0,1,9,1,0,1,21.39,...,1,2,1,0,0,0,0,0,1,2020-01-01 14:00:00
4,3,1,0,1,2,12,0,0,0,20.15,...,1,1,1,0,0,0,0,0,0,2020-01-01 08:00:00


In [74]:
df


,specialty,gender,no_show,disability,appointment_shift,age,under_12_years_old,over_60_years_old,patient_needs_companion,average_temp_day,...,storm_day_before,rain_intensity,heat_intensity,Hipertension,Diabetes,Alcoholism,Handcap,Scholarship,SMS_received,appointment_datetime
0,1,0,1,0,1,9,1,0,1,23.18,...,1,1,3,0,0,0,0,0,0,2020-01-01 17:00:00
1,1,1,0,0,2,11,1,0,1,14.31,...,1,1,2,0,0,0,0,0,0,2020-01-01 07:00:00
2,2,1,0,0,1,8,1,0,1,21.61,...,1,1,3,0,0,0,0,0,0,2020-01-01 16:00:00
3,2,1,1,0,1,9,1,0,1,21.39,...,1,2,1,0,0,0,0,0,1,2020-01-01 14:00:00
4,3,1,0,1,2,12,0,0,0,20.15,...,1,1,1,0,0,0,0,0,0,2020-01-01 08:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109540,1,1,1,0,1,18,0,0,0,18.97,...,1,1,1,0,0,0,0,0,0,2021-05-11 16:00:00
109541,2,0,1,1,2,18,0,0,0,24.46,...,1,2,3,0,0,0,0,0,0,2021-05-11 09:00:00
109542,1,0,0,0,1,8,1,0,1,20.88,...,1,2,1,0,0,0,0,0,1,2021-05-11 13:00:00
109543,3,0,0,1,2,7,1,0,1,24.89,...,1,1,3,0,0,0,0,0,0,2021-05-11 08:00:00


# Feature Engineering from appointment_datetime

In [75]:
df['appointment_datetime'] = pd.to_datetime(
    df['appointment_datetime'],
    errors='coerce'
)

In [76]:
print(df['appointment_datetime'].dtype)
print(df['appointment_datetime'].isnull().sum())

datetime64[ns]
0


In [77]:

df['appointment_day'] = df['appointment_datetime'].dt.day

df['appointment_month'] = df['appointment_datetime'].dt.month

df['appointment_dayofweek'] = df['appointment_datetime'].dt.dayofweek

df['appointment_hour'] = df['appointment_datetime'].dt.hour

df['appointment_weekday'] = (
    df['appointment_datetime'].dt.day_name().str.lower()
)

df['is_weekend'] = (
    df['appointment_dayofweek'] >= 5
).astype(int)

In [78]:
def get_time_period(hour):

    if hour < 12:
        return 'morning'

    elif hour < 17:
        return 'afternoon'

    else:
        return 'evening'


df['appointment_period'] = (
    df['appointment_hour'].apply(get_time_period)
)

In [79]:
def age_group(age):

    if age < 18:
        return 'under_18'

    elif age < 30:
        return '18_29'

    elif age < 45:
        return '30_44'

    elif age < 60:
        return '45_59'

    else:
        return '60_plus'


df['age_group'] = df['age'].apply(age_group)

In [80]:
df.select_dtypes(include='object').columns

Index(['appointment_weekday', 'appointment_period', 'age_group'], dtype='object')

In [81]:
categorical_cols = [
    'appointment_shift',
    'rain_intensity',
    'heat_intensity',
    'appointment_weekday',
    'appointment_period',
    'age_group'
]

df = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

In [82]:
df.shape

(109545, 46)

In [83]:
df.dtypes.value_counts()

int64             37
float64            4
int32              4
datetime64[ns]     1
Name: count, dtype: int64

In [84]:
df.select_dtypes(include='object').columns

Index([], dtype='object')

In [85]:
# Features
X = df.drop(columns=['no_show', 'appointment_datetime'])

# Target
y = df['no_show']

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (109545, 44)
y shape: (109545,)


In [86]:
print(y.value_counts())

no_show
0    74714
1    34831
Name: count, dtype: int64


In [87]:
print("Total missing values:", X.isnull().sum().sum())

Total missing values: 0


In [88]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (87636, 44)
X_test : (21909, 44)
y_train: (87636,)
y_test : (21909,)


In [89]:
print("Train:")
print(y_train.value_counts(normalize=True))

print("\nTest:")
print(y_test.value_counts(normalize=True))

Train:
no_show
0    0.682037
1    0.317963
Name: proportion, dtype: float64

Test:
no_show
0    0.682048
1    0.317952
Name: proportion, dtype: float64


In [90]:

negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

scale_pos_weight = negative / positive

print("Negative class:", negative)
print("Positive class:", positive)
print("Scale Pos Weight:", scale_pos_weight)

Negative class: 59771
Positive class: 27865
Scale Pos Weight: 2.145020635205455


In [91]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    )
}

In [92]:

for name, model in models.items():

    print("\nTraining:", name)

    model.fit(X_train, y_train)

    print(name, "trained successfully!")


Training: Logistic Regression


c:\Users\singh\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression trained successfully!

Training: Random Forest
Random Forest trained successfully!

Training: XGBoost
XGBoost trained successfully!


In [93]:
results = []

for name, model in models.items():

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        'Model': name,
        'F1 Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results)

results_df

,Model,F1 Score,ROC-AUC,Precision,Recall
0,Logistic Regression,0.500195,0.665334,0.456615,0.552972
1,Random Forest,0.396672,0.746508,0.594099,0.297732
2,XGBoost,0.589286,0.755157,0.527192,0.667959


In [94]:
df

,specialty,gender,no_show,disability,age,under_12_years_old,over_60_years_old,patient_needs_companion,average_temp_day,average_rain_day,...,appointment_weekday_sunday,appointment_weekday_thursday,appointment_weekday_tuesday,appointment_weekday_wednesday,appointment_period_evening,appointment_period_morning,age_group_30_44,age_group_45_59,age_group_60_plus,age_group_under_18
0,1,0,1,0,9,1,0,1,23.18,0.00,...,0,0,0,1,1,0,0,0,0,1
1,1,1,0,0,11,1,0,1,14.31,0.02,...,0,0,0,1,0,1,0,0,0,1
2,2,1,0,0,8,1,0,1,21.61,0.01,...,0,0,0,1,0,0,0,0,0,1
3,2,1,1,0,9,1,0,1,21.39,0.11,...,0,0,0,1,0,0,0,0,0,1
4,3,1,0,1,12,0,0,0,20.15,0.02,...,0,0,0,1,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109540,1,1,1,0,18,0,0,0,18.97,0.00,...,0,0,1,0,0,0,0,0,0,0
109541,2,0,1,1,18,0,0,0,24.46,0.23,...,0,0,1,0,0,1,0,0,0,0
109542,1,0,0,0,8,1,0,1,20.88,0.00,...,0,0,1,0,0,0,0,0,0,1
109543,3,0,0,1,7,1,0,1,24.89,0.00,...,0,0,1,0,0,1,0,0,0,1


In [95]:
# Select best model based on F1 Score
best_model_name = results_df.loc[
    results_df['F1 Score'].idxmax(),
    'Model'
]

best_model = models[best_model_name]

# Threshold selected from threshold tuning
best_threshold = 0.45

# Save model package
model_package = {
    "model": best_model,
    "threshold": best_threshold,
    "features": X.columns.tolist()
}

joblib.dump(
    model_package,
    "models/best_no_show_model.pkl"
)

print("Best Model:", best_model_name)
print("F1 Score:", results_df['F1 Score'].max())
print("Threshold:", best_threshold)
print("Model saved successfully!")

Best Model: XGBoost
F1 Score: 0.5892857142857143
Threshold: 0.45
Model saved successfully!


#---